# Intro

In this jupyter notebook I will try to break down the python scripts (mutation caller) wrote by Eva and go throuth them step-by-step. This is to fisrt of all understand each step and procedure but also to fact-check the results.

In [19]:
# load packages

import os
import sys
import re
import pysam
#import fasta_tools as ft
import numpy as np
np.set_printoptions(threshold=sys.maxsize, linewidth=sys.maxsize, precision=2, suppress=True)
import time
from collections import Counter
from tqdm import tqdm
import pandas as pd

### break-down of input files


In [6]:

# assign paths to: 1- alignment file of each sample and 2- reference sequence and name of the contig of the reference.

alignment_file = "/home/amovas/data/genome-evo-proj/data/processed-data/mappings/3-p/iv/20/20MT2EXPIVVP180seq09052019_S24_L001_aligned.bam"      # sys.argv[1]
ancestor_file = "/home/amovas/data/genome-evo-proj/data/reference/plasmid/hiv_plasmid_ref_genome.fasta"   #sys.argv[2]
contig_name = "AF324493.2" #"NL43seq210314" #"NL43_ann_wk0virusPassRef_SEQregion"   #"K03455.1"    #sys.argv[3]

# in the following lines try to extract the sample name and passage number out of the absolute path to the alignment file
fields = alignment_file.split('/')
sample = fields[-1]

passage = re.findall(r'VP[0-9]+',sample)[-1][2:]

# not necessary to run this, Basically this is a check to see if the sample has already been processed or not. I think this check is already embedded in snakemake functionality.
try:
    with open('mutations/backup/{}/{}/{}.csv'.format(fields[1],fields[2],passage)) as f:
        csv_here = f.readlines()
    still_to_do = False
except:
    still_to_do = True

# do not really know what is the function of extra_info object atm!
# I think this is to get the line and passage number. However, to get the line number the indext must be -2 and not 2
extra_info = fields[-2]+','+passage


print(passage, sample, extra_info)




180 20MT2EXPIVVP180seq09052019_S24_L001_aligned.bam 20,180


In [7]:



# read in the ancestor file
with open(ancestor_file) as f:
    ancestor = f.readlines()

ancestor = ancestor[1][:-1] # remove the new line character
start = 0
stop = len(ancestor)

#ancestor = ancestor[start:stop]

# read in the alignment file
samfile = pysam.AlignmentFile(alignment_file)


# coverage is a two dimensional array. The first dimension is of length 4 and each corresponds to a nucleotide base (ACGT order). The second dimension corresponds to the position in the reference genome
# count_coverage is a method of class pysam.libcalignmentfile.AlignmentFile
all_coverage = np.array(samfile.count_coverage(contig_name, start=start,
                                                       stop=stop,quality_threshold=35)) # quality threshold must be consistent throughout the analysis

# print(all_coverage[:,3000:3010])
# print(ancestor[3000:3010])



### break-down of 1st function

The  only thing that I changed is to report the reference base as well as the alternative (mutation) base!

In [25]:
#params of the function:

coverage = all_coverage
min_coverage = 1000


In [26]:
# check if the coverage is high enough. if not set the values to zero for easier downstream handling of the data
too_low_coverage = coverage.sum(0) < min_coverage # equivalent to sum(all_coverage,0)
coverage[:,too_low_coverage] = 0




In [27]:
# calculate the fraction of each base recoreded for each position
total_coverage = coverage.sum(0)

fractions = 100*coverage/total_coverage

fractions[:,3000:3010]


/tmp/ipykernel_3958909/2941643197.py:4: RuntimeWarning: invalid value encountered in divide
  fractions = 100*coverage/total_coverage


array([[ 99.96,   0.06,   0.04,   0.04,  99.94,   0.  ,   0.08,   0.04,  99.93,  99.97],
       [  0.  ,   0.  ,   0.  ,   0.  ,   0.  ,   0.  ,   0.  ,   0.  ,   0.  ,   0.  ],
       [  0.03,  99.92,  99.96,  99.94,   0.05,   0.  ,  99.91,  99.96,   0.05,   0.03],
       [  0.  ,   0.01,   0.  ,   0.01,   0.  , 100.  ,   0.  ,   0.  ,   0.02,   0.01]])

In [28]:
# Convert the reference sequence to index-based sequence
bases = 'ACGT'
ref_bases = [bases.find(i) for i in ancestor]

print(ancestor[1:10])
print(ref_bases[1:10])

GGAAGGGCT
[2, 2, 0, 0, 2, 2, 2, 1, 3]


In [29]:
# set fraction to zero for reference bases

fractions[ref_bases,range(len(ancestor))] = 0



In [30]:

#find the mutations with high enough fractions (we're not interested in low-frequency mutations)
min_fractions = 0.001
#return_muts = []
muts = np.where(fractions>min_fractions)

# muts is a tuple of two numpy arrays. the first nparray element stores the index of first array dimension and the second tuple the indices of the second dimension
print(type(muts))


<class 'tuple'>


In [32]:

# determine mutations and collect them in a list with relevant information
muts_to_return = []
for new, loc in zip(muts[0],muts[1]):
    fraction = fractions[new,loc]
    n_changed = coverage[new,loc]
    ref_base = ancestor[loc] #include the reference base in the final output
    coverage_here = total_coverage[loc]
    if coverage_here>min_coverage:
        muts_to_return.append(('M',loc,ref_base,bases[new],fraction,n_changed,coverage_here))

print(muts_to_return[0])
print(type(muts_to_return[0]))

df = pd.DataFrame(muts_to_return, columns = ["type", "pos", "ref", "alt", "frac", "reads", "coverage"])

df[df["frac"] >= 99]


('M', 477, 'T', 'A', 0.06523157208088715, 1, 1533)
<class 'tuple'>


,type,pos,ref,alt,frac,reads,coverage
48,M,632,C,A,100.000000,13420,13420
110,M,751,G,A,99.827760,16808,16837
139,M,807,G,A,99.963603,16479,16485
180,M,905,G,A,99.926686,17719,17732
333,M,1282,G,A,99.928020,20824,20839
447,M,1520,G,A,99.967222,18299,18305
519,M,1709,G,A,99.905309,15826,15841
623,M,1961,G,A,100.000000,15576,15576
715,M,2180,G,A,99.969765,16532,16537
821,M,2401,T,A,99.927361,34392,34417


### Break-down of the second function (find_large_dels)

Main error and modification was related to chimeric read reference position that was 1-based while had to be in 0-based coordination!

In [33]:
# params

start=0
stop=None
threshold=5 # note that threshold is defined very arbitrarily. Maybe it would be wise to take a look at (read_end_offset + chimera_start_offset - read_length) == (read_alignment_length + chimera_alignment_length - read_length)
# distribution for all reads before setting a hard cut-off.

In [34]:

# this is optional probably to limit the deletion detection to a section of a genome if stop < reference length
if stop is not None:
    size = stop-start
else:
    size = samfile.lengths[0] # an attribute of the samfile that gets the length of reference sequence used for mapping

print(size)

stop = 

14825


In [35]:
## read object of class "pysam.libcalignedsegment.AlignedSegment" has so many attributes. The description of some of the attributes:


counter = 0
for read in samfile.fetch(contig=contig_name,start=start,stop=stop): # fetch is a method of samfile that fetched reads in the specified region by the arguments
    counter += 1
    if counter < 6:
        print(read)
        print(read.mapping_quality) # the quality of the mapping.
        print(read.inferred_length) # the length of the read (actual, according to cigarstring)
        print(read.cigarstring) # literal cigar string
        print(read.cigartuples) # index-based cigar
        print(read.tags) # reads tags
        print(read.get_reference_positions(full_length=True)) # this method returns positions of the reference that read is aligned to (full_length if set, retuns None for H/S clipped section)
        print(read.reference_start) # 0-based position of the left-most base of the reference aligned to the read (if hard/soft clip at the start of read this will not correspond to first base of read)
        print(read.reference_end) # 0-based position of the right-most base of the reference aligned to the read (if hard/soft clip at the end of read this will not correspond to last base of read)
    else:
        break

M02081:386:000000000-C8BWK:1:1102:19926:21648	163	#0	37	0	50M	#0	37	50	TCCTTGATCTGTGGATCTACCACACACAAGGCTACTTCCCTGATTGGCAG	array('B', [34, 34, 34, 35, 34, 37, 37, 37, 37, 37, 37, 37, 38, 38, 38, 38, 38, 38, 38, 38, 38, 38, 39, 39, 38, 38, 38, 38, 38, 39, 38, 39, 39, 39, 37, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 37, 39, 38, 39, 38])	[('NM', 0), ('MD', '50'), ('MC', '50M'), ('AS', 50), ('XS', 50), ('XA', 'AF324493.2,+9112,50M,0;')]
0
50
50M
[(0, 50)]
[('NM', 0), ('MD', '50'), ('MC', '50M'), ('AS', 50), ('XS', 50), ('XA', 'AF324493.2,+9112,50M,0;')]
[36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85]
36
86
M02081:386:000000000-C8BWK:1:1106:10874:23621	163	#0	37	0	44M	#0	37	44	TCCTTGATCTGTGGATCTACCACACACAAGGCTACTTCCCTGAT	array('B', [32, 32, 30, 32, 32, 35, 37, 37, 33, 37, 37, 35, 38, 38, 37, 38, 38, 38, 38, 38, 38, 38, 38, 39, 38, 38, 38, 38,

In [36]:
## Chimeric reads are also called split reads. They are indicative of structural variation.
def parse_cigar(cigar): # this function takes the cigar string (e.g., read.cigarstring or chimeric cigar string extracted from reads) and returns in a format exactly similar to read.cigar
    operations = re.findall('([0-9]+)([A-Z]+)', cigar)
    numbers = {'M':0,'I':1,'D':2,'N':3,'S':4,'H':5,'P':6,'=':7,'X':8,'b':9}
    return [(numbers[i[1]],int(i[0])) for i in operations] # this one liner is the compact form of the following:

'''
dd = []
for i in operations:
    dd.append((numbers[i[1]],int(i[0])))
return dd
'''



'\ndd = []\nfor i in operations:\n    dd.append((numbers[i[1]],int(i[0])))\nreturn dd\n'

In [37]:
# in the following chunk of code we detect deletions that are represented as soft/hard clips in the cigar strings
counter = 300
quals = []
for read in samfile.fetch(contig=contig_name,start=start,stop=stop):
    if read.mapping_quality>30:
        if 'H' in read.cigarstring or 'S' in read.cigarstring:
            counter += 1
            if counter < 1700:
                #print(read.inferred_length) # read length based on cigar string (deletions and insertions are counted, a.k.a actual read length, not read length according to start/end mapping location on the reference genome)
                deletion = None
                #already get the real starts and ends of this read (not only where mapping starts and ends)
                start_offset = read.cigar[0][1] if read.cigar[0][0] in [4,5] else 0
                end_offset = read.cigar[-1][1] if read.cigar[-1][0] in [4,5] else 0
                read_start = read.reference_start-start_offset # note that reference_start property is the 0-based leftmost coordinate (while the sam stores information on a 1-based system)
                
                read_end = read.reference_end+end_offset
                read_length = sum([i[1] for i in read.cigar])
                tags = dict(read.tags) # SA tags present information about the chimeric. Of note, the NM tag depicts the number of mismatches + inserted and deleted bases
                if 'SA' in tags.keys():
                    fields = tags['SA'].split(',') #the SA tag holds information on chimeric reads (contig_name, ref-start-pos, strand, cigar-string, mapping quality, NM)
                    if int(fields[4])>30:
                        #print(fields)
                        #print(read)
                        
                        chimera_cigar = parse_cigar(fields[3])
                        
                        
                        # if the cigarstring starts with a hard or soft clip, get the length of the clip
                        chimera_start_offset = chimera_cigar[0][1] if chimera_cigar[0][0] in [4,5] else 0 
                        # if the cigarstring ends with a hard or soft clip, get the length of the clip
                        chimera_end_offset = chimera_cigar[-1][1] if chimera_cigar[-1][0] in [4,5] else 0
                        
                        chimera_start_ref = int(fields[1]) - 1 # extract the reference start position from the SA tag (note that the it has to be converted to 0-based coordinates which in Eva's script was missed)
                        chimera_start = chimera_start_ref - chimera_start_offset
                        chimera_end = chimera_start + sum([i[1] for i in chimera_cigar]) # get the end of chimera
                        chimera_end_ref = chimera_end-chimera_end_offset
                        chimera_len = chimera_end - chimera_start # not used, can be skipped
                        
                        
                        
                        if end_offset!=0 and chimera_start_offset !=0: #original is clipped at end, chimera at beginning
                            if read.reference_end < chimera_start_ref: #the order of the mappings is right
                                if abs(end_offset + chimera_start_offset-read_length) < threshold:
                                    deletion = (read.reference_end,chimera_start_ref)
                                    read_end = chimera_end #end of the read, for potentially marking later dels
                                    # end_offset = chimera_end_offset

                        if start_offset!=0 and chimera_end_offset!=0: #chimera is clipped at end, original at begining
                            if chimera_end_ref < read.reference_start:#order is right
                                if abs(end_offset + chimera_start_offset-read_length) < threshold:
                                    deletion = (chimera_end_ref,read.reference_start)
                                    read_start = chimera_start 
                                    # start_offset = chimera_start_offset
                        
                        # the following print lines are for inspection of the steps
                        if deletion is not None:
                            print(abs(end_offset + chimera_start_offset-read_length)) 
                            print(read)
                            # print(f"read_length is {read_length}")
                            # print(f"read_start is {read_start} and read_end is {read_end}")
                            # print(f"start_offset is {start_offset} and end_offset is {end_offset}")
                            print(f"read_start_reference is {read.reference_start} and read_end_reference is {read.reference_end}")
                        
                        
                            # print(f"chimera_start is {chimera_start} and chimera_end is {chimera_end}")
                            # print(f"chimera_start_offset is {chimera_start_offset} and chimera_end_offset is {chimera_end_offset}")
                            print(f"chimera_start_ref is {chimera_start_ref} and chimera_end_ref is {chimera_end_ref}")
                            
                        
                            print(f"!!! Deletion is {deletion}")
                            



1
M02081:386:000000000-C8BWK:1:2109:26013:17721	163	#0	469	60	19M1I177M37S	#0	469	196	GACCAGATCTGAGCCTGGGAAGCTCTCTGGCTAACTAGGGAACCCACTGCTTAAGCCTCAATAAAGCTTGCCTTGAGTGCTCAAAGTAGTGTGTGCCCGTCTGTTGTGTGACTCTGGTAACTAGAGATCCCTCAGACCCTTTTAGTCAGTGTGGAAAATCTCTAGAAGTGGCGCCCGAACAGGGACTTGAAAGCGAACAGGATCAGAAGAACTTAGATCATTATATAATACAAT	array('B', [33, 33, 32, 32, 32, 37, 37, 37, 37, 37, 37, 37, 38, 38, 38, 38, 38, 38, 38, 37, 38, 38, 38, 39, 34, 38, 39, 39, 35, 37, 37, 39, 39, 39, 39, 39, 38, 39, 37, 39, 39, 39, 38, 38, 38, 38, 38, 39, 39, 39, 38, 39, 39, 39, 38, 39, 39, 39, 39, 39, 18, 37, 36, 37, 38, 37, 39, 39, 39, 38, 39, 39, 38, 39, 37, 39, 39, 38, 39, 39, 39, 39, 37, 39, 39, 39, 31, 37, 39, 38, 37, 37, 38, 39, 39, 39, 38, 39, 38, 36, 38, 39, 36, 38, 39, 37, 30, 36, 38, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 39, 38, 38, 39, 39, 37, 39, 39, 39, 39, 38, 39, 36, 39, 39, 39, 39, 39, 39, 38, 39, 37, 39, 39, 38, 39, 39, 37, 33, 38, 39, 39, 39, 39, 39, 39, 37, 37,

In [38]:
print("read:"+ ancestor[(788):(789)])

print("chimera: " + ancestor[(6220):(6238)])


rr = "GATCCCTCAGAACCTTATAGTTAGTACGAAAAATCTCTAGCAGTGGCGCCCGAACAGGGACTCGAAAGCGAAAGTAAAGCCAGAGAAGATCTCTCGACGCAGGACTCGGCTTGCTGAAGCGCGCACGGCAAGAGGCGAGGGGCGGCGACTGGTGAGTACGCCAAAATTGTGACTAGCGGAGGCTAGAAGGAGAAATATCAGCACTTGTGGAGATGGGGGTGGAAATGGGGCACCATGCTCCTTGGGATATT"
print(len(rr))
print(rr[184:204])

read:G
chimera: ATGAGAGTGAAGGAGAAG
251
AGAAGGAGAAATATCAGCAC


In [40]:
counts_dels = Counter(dels) # counts the items, in this case tuples with identical start/end deletion position
dels = Counter() # create a counter that can count iterable items like below 
for deletion in counts_dels:
    if counts_dels[deletion]>10:
        dels[('D',deletion[0],deletion[1])]+=1


NameError: name 'counts_dels' is not defined

### Break-down of the third function

first mistake is that when there is a D in cigar string it should not be added to "location" variable that is later used to subset the read, since the deletions are not part of the read.

0-based system coordinate for specifying mutations, insertions, and deletions: https://www.biostars.org/p/84686/

Further readins: https://tidyomics.com/blog/2018/12/09/2018-12-09-the-devil-0-and-1-coordinate-system-in-genomics/

In [41]:
# Params

samfile # alignment file loaded via pysam,AlignmentFile
coverage=all_coverage # coverage of the genome calculated via amfile.count_coverage
contig_name
muts=None
n=None
start=0
stop=None
overall_overlap=5
min_coverage=1000

In [42]:
if muts is None:
        muts = Counter()
if n is not None:
    counter = 0

print(muts)

Counter()


In [43]:
starts = [i[1] for i in muts if i[0] == 'D']
ends = [i[2] for i in muts if i[0] == 'D']



In [ ]:
counter = 0


for read in samfile.fetch(contig=contig_name,start=start,stop=stop):
        if counter < 40:
            
            counter += 1
            #print(read.cigarstring)
            #print(read.inferred_length)
            cigar = read.cigar
            location = 0
            for operation in cigar:
                #print(operation)
                if operation[0]!=5 and operation[0]!=2: #hard clipping, not in query sequence #!!! and not D
                    location += operation[1]
                if operation[0] == 0: #match
                    pass
                # elif operation[0] == 1: #insertion
                #     insert = read.query_sequence[location:location+operation[1]]
                #     map_pos = read.reference_start
                #     muts[('I',map_pos+location,insert)]+=1
                    
                #     print(muts)
                #     print(read)
                #     print(read.query_sequence[location-1:location+operation[1]+1])
                #     print(ancestor[read.reference_start:(read.reference_start+location+operation[1]+1)])
                #     print(read.cigarstring)
                #     print(location)
                    
                elif operation[0] == 2:
                    map_pos = read.reference_start
                    delstart = map_pos+location
                    delend = delstart+operation[1]
                    muts[ ('D',delstart,delend)]+=1
                    
                    print(muts)
                    print(read)
                    print(ancestor[read.reference_start:read.reference_start+20])
                    print(ancestor[delstart:delend])
                    #print(read.cigarstring)
                    #print(location)
                    
            muts = Counter()
        else: 
            break



In [53]:

muts = Counter()
for read in samfile.fetch(contig=contig_name,start=start,stop=stop):
    if read.query_name == "M02081:386:000000000-C8BWK:1:1109:18680:8686": # filter on mapping quality was missing in eva's version!
        print(read)
       
        #print(read.cigarstring)

        if "I" in read.cigarstring:
            print(read)
            print(read.cigarstring)
            #print(read.inferred_length)
            cigar = read.cigar
            read_location = 0
            reference_location = read.reference_start
            for operation in cigar:
                if operation[0] == 4:
                    read_location += operation[1]
                elif operation[0] == 2:
                    reference_location += operation[1]
                    
                    
                    delstart = reference_location-operation[1]
                    delend = reference_location
                    deletion = ancestor[delstart:delend]
                    muts[ ('D',delstart,delend,deletion)]+=1
                    
                    
                elif operation[0] == 1:
                    read_location += operation[1]
                    
                    insert = read.query_sequence[read_location:read_location+operation[1]]
                    reference_start = read.reference_start
                    muts[('I',reference_start+reference_location,insert)]+=1
                    
                    print(muts)
                    
                elif operation[0] == 0:
                    read_location += operation[1]
                    reference_location += operation[1]
                    
                print(read_location,reference_location)
                
    else:
        break



In [ ]:

# comparing the two functions

def my_indel_function(samfile, coverage, contig_name,counter_lim):
    counter = 0
    muts=Counter()
    for read in samfile.fetch(contig=contig_name,start=start,stop=stop):
        if read.mapping_quality > 30: # filter on mapping quality was missing in eva's version!
            counter += 1
            if counter < counter_lim:
                cigar = read.cigar
                read_location = 1
                reference_location = read.reference_start
                for operation in cigar:
                    if operation[0] == 4:
                        read_location += operation[1]
                    if operation[0] == 2:
                        
                        
                        
                        delstart = reference_location + 1
                        delend = reference_location + operation[1] + 1
                        deletion = ancestor[delstart:delend]
                        muts[ ('D',delstart,delend,deletion)]+=1
                        
                        reference_location += operation[1]
                        
                        
                        
                        
                    if operation[0] == 1:
                        
                        
                        insert = read.query_sequence[read_location:read_location+operation[1]]
                        reference_start = read.reference_start
                        muts[('I',reference_start+read_location,reference_start+read_location,insert)]+=1
                        read_location += operation[1]
                        
                        
                        #print(muts)
                        
                    if operation[0] == 0:
                        read_location += operation[1]
                        reference_location += operation[1]
                        
                    #print(read_location,reference_location)
            else:
                return(muts)

def eva_indel_function(samfile, coverage,contig_name, counter_lim):
    counter = 0
    muts=Counter()
    for read in samfile.fetch(contig=contig_name,start=start,stop=stop):
        if read.mapping_quality > 30:
            counter += 1
            if counter < counter_lim:
                cigar = read.cigar
                location = 0
                for operation in cigar:
                    if operation[0]!=5: #hard clipping, not in query sequence
                        location += operation[1]
                    if operation[0] == 0: #match:
                        pass
                    elif operation[0] == 1: #insert
                        insert = read.query_sequence[location:location+operation[1]]
                        map_pos = read.reference_start
                        muts[('I',map_pos+location,insert)]+=1
                        
                        
                    elif operation[0] == 2: #deletion
                        map_pos = read.reference_start
                        delstart = map_pos+location
                        delend = delstart+operation[1]
                        muts[ ('D',delstart,delend)]+=1
                        
            else:
                return(muts)

counter_lim = 13700

my_muts = my_indel_function(samfile, coverage, contig_name, counter_lim)

eva_muts = eva_indel_function(samfile, coverage, contig_name, counter_lim)

print(Counter(my_muts.values()))
print(sum(my_muts.values()))
# how to convert dict_keys/dict_values to list or np array? then count (like table function in R)?
print("_______")
print(Counter(eva_muts.values()))
print(sum(eva_muts.values()))




In [67]:
muts = Counter()
counter = 0
for read in samfile.fetch(contig=contig_name,start=start,stop=stop):
    counter += 1
    if read.query_name == "M02081:386:000000000-C8BWK:1:1109:18680:8686":
        cigar = read.cigar
        print(cigar)
        read_location = 0
        reference_location = read.reference_start
        for operation in cigar:
            if operation[0] == 4:
                read_location += operation[1]
                
            if operation[0] == 0:
                read_location += operation[1]
                reference_location += operation[1]
                
            elif operation[0] == 1:
                insert = read.query_sequence[read_location:read_location+operation[1]]
                reference_start = read.reference_start
                muts[('I',reference_start+read_location,reference_start+read_location,'-',insert)]+=1
                
                read_location += operation[1]
            
            elif operation[0] == 2:
                delstart = reference_location + 1
                delend = reference_location + operation[1] + 1
                deletion = ancestor[delstart:delend]
                muts[ ('D',delstart,delend,deletion,'-')]+=1
                
                reference_location += operation[1]
                
            elif operation[0] == 4 or operation[0] == 5: #clip (soft or hard)
                for offset in range(-overall_overlap,overall_overlap):
                    if read.reference_start+offset in ends:
                        for dele in [i for i in muts if ((i[0] == 'D') and (i[2] == read.reference_start+offset))]:
                            muts[dele]+=1
                    if read.reference_end+offset in starts:
                        for dele in [i for i in muts if ((i[0] == 'D') and (i[1] == read.reference_end+offset))]:
                            muts[dele]+=1
            
            print(read_location,reference_location)
            print(muts)

[(0, 184), (1, 2), (0, 65)]
184 759
Counter()
186 759
Counter({('I', 759, 759, '-', 'AT'): 1})
251 824
Counter({('I', 759, 759, '-', 'AT'): 1})
[(0, 250)]
250 1024
Counter({('I', 759, 759, '-', 'AT'): 1})
